# Batch processing of Sentinel-2 images

This notebook processes a time series of Level-1 images (Sentinel-2 L1C `.SAFE`, or Landsat
Collection 2 L1) of one or several tiles into GRS Level-2A products, with GRS 2.2.0:

1. **Configuration**: input/output folders, tiles, date range, auxiliary data, processing options, and
   check of the look-up tables set in `grs/config.yml`.
2. **List the Level-1 images** found on disk, with their cloud cover, and select a date range.
3. **Auxiliary data and output naming**: CAMS file of the month, optional DEM of the tile, and one
   L2A product per image, named as with the `grs` command line.
4. **Run the batch**: images already processed are skipped, too cloudy images or images without CAMS
   file are skipped, a failing image is logged and does not stop the batch, and a summary table is saved.
5. **Quick look** at one L2A product.

Each image is processed with {py:class}`grs.Process <grs.grs_process.Process>` as in the
{doc}`basic tutorial <../basics>`. For long batches, the command line is more robust than a notebook: each
image runs in its own Python process, so memory is fully released between images (see the last section).

**Resources**: with GRS 2.2.0, a full Sentinel-2 tile at 20 m takes about 8 minutes and up to 30 GB of
memory (measured on tile 31TEJ with 8 threads, `processor: ncpu` of `grs/config.yml`); at 60 m, about 9
times fewer pixels. Starting Jupyter with the environment variable `MALLOC_ARENA_MAX=2` lets the memory
freed by the processing threads be reused from one image to the next.

In [ ]:
import os
import gc
import re
import glob
import time
import logging
import warnings
import datetime as dt
from pathlib import Path
from importlib.resources import files

import numpy as np
import pandas as pd
import xarray as xr
import yaml
import matplotlib.pyplot as plt

import grs

opj = os.path.join

# INFO messages of the processing chain; set to logging.WARNING for a quieter batch
logging.getLogger().setLevel(logging.INFO)
# deprecation notices of the dependencies, not useful here
warnings.filterwarnings('ignore', category=FutureWarning)

print(f'grs: {grs.__version__}')

## 1. Configuration

The Level-1 images are expected in a `<idir>/<tile>/YYYY/MM/DD/<image>` tree; the L2A products are
written to the same tree under `<odir>/<tile>`.

| Parameter | Description |
|---|---|
| `idir`, `odir` | root folders of the Level-1 inputs and of the L2A outputs |
| `tiles` | tile sub-folders to process (e.g. `30TWT`, or a Landsat path/row `199027`) |
| `pattern` | glob pattern of the Level-1 image names (`'S2*.SAFE'`, `'L*_L1*'` for Landsat) |
| `start_date`, `end_date` | date range to process (inclusive) |
| `cams_file_template` | CAMS file of the month of the acquisition, formatted with the image `date` (mandatory, see the {doc}`CAMS tutorial <../auxdata>`) |
| `dem_file_template` | DEM GeoTIFF subset to the tile, formatted with `tile`; `None` or missing file: no DEM (fine over the sea) |
| `resolution` | pixel size (m) of the L2A product (10, 20 or 60 for Sentinel-2, 30 for Landsat) |
| `max_cloud_cover` | skip the images whose Level-1 cloud cover (0-1) is greater or equal |
| `opac_model`, `scale_aot` | force the aerosol model (`None`: from CAMS), scaling factor of the CAMS AOD |
| `levname`, `suffix` | level name and suffix of the output name (`None`: `_V<grs version>`), as `--levname` / `--suffix` of the command line |
| `overwrite` | reprocess images whose L2A output already exists |

The rendered run below processes the images of 1 to 4 August 2025 of tile 31TEJ (Gulf of Lion: Thau
lagoon and the coast of Montpellier).

In [ ]:
idir = '/data/satellite/Sentinel-2/L1C'
odir = '/data/satellite/Sentinel-2/L2A_nc'
tiles = ['31TEJ']
pattern = 'S2*.SAFE'
start_date, end_date = '2025-08-01', '2025-08-04'

# other example (Landsat, path/row sub-folders):
# idir, odir, tiles, pattern = '/data/satellite/landsat/L1', '/data/satellite/landsat/L2A', ['199027'], 'L*_L1*'

cams_file_template = '/data/cams/world/cams_forecast_{date:%Y-%m}.nc'
dem_file_template = '/home/harmel/Dropbox/Dropbox/satellite/dem/COP-DEM_GLO-30-DGED_{tile}.tif'   # or None

resolution = 20        # m
max_cloud_cover = 0.6  # 0-1
opac_model = None      # e.g. 'DESE_rh70' to force desert dust
scale_aot = 1

levname = 'L2AGRS'
suffix = None          # None: '_V<grs version>'
overwrite = False

The radiative transfer look-up tables (LUT) are read from the `grsdata` folder set in the `config.yml`
file of the installed package (see {doc}`../../installation`). Check that they are found before starting
a long batch:

In [ ]:
config_file = files('grs') / 'config.yml'
with open(config_file) as f:
    config = yaml.safe_load(f)['path']
print(f'configuration: {config_file}')
for key in ('toa_lut', 'trans_lut'):
    lut = opj(config['grsdata'], config[key])
    print(f"{key:9s} {lut}: {'found' if os.path.exists(lut) else 'MISSING, edit grsdata in config.yml'}")

## 2. List the Level-1 images

The images are indexed by their acquisition date, taken from the `YYYY/MM/DD` folders, so that a date
range can be selected with `.loc`. The cloud cover is read from the Level-1 metadata
(`MTD_MSIL1C.xml` for Sentinel-2, `*_MTL.txt` for Landsat); it is `NaN` when it cannot be found.

In [ ]:
def cloud_cover(path):
    """Level-1 cloud cover (0-1) from the image metadata, NaN if not found."""
    patterns = [('MTD_MSIL1C.xml', r'<Cloud_Coverage_Assessment>\s*([\d.]+)'),
                ('*_MTL.txt', r'\bCLOUD_COVER\s*=\s*"?([\d.]+)')]
    for metadata, regex in patterns:
        for file in glob.glob(opj(path, metadata)):
            with open(file) as f:
                match = re.search(regex, f.read())
            if match:
                return float(match.group(1)) / 100
    return np.nan


def list_images(idir, tiles, pattern='S2*.SAFE'):
    """Table of the Level-1 images found in <idir>/<tile>/YYYY/MM/DD/, indexed by acquisition date."""
    rows = []
    for tile in tiles:
        for path in sorted(glob.glob(opj(idir, tile, '*', '*', '*', pattern))):
            year, month, day = Path(path).parts[-4:-1]
            rows.append(dict(date=pd.Timestamp(f'{year}-{month}-{day}'), tile=tile,
                             name=os.path.basename(path.rstrip('/')), path=path))
    df = pd.DataFrame(rows, columns=['date', 'tile', 'name', 'path'])
    df['cloud_cover'] = df['path'].map(cloud_cover)
    return df.set_index('date').sort_index()


images = list_images(idir, tiles, pattern)
print(f'{len(images)} Level-1 images found in {idir} for tiles {tiles}')
images.head()

In [ ]:
selection = images.loc[start_date:end_date]
print(f'{len(selection)} images between {start_date} and {end_date}')
selection.groupby([selection.tile, selection.index.year]).size().rename('images per year').unstack(fill_value=0)

## 3. Auxiliary data and output naming

For each image:

- the **CAMS file** is the one of the month of the acquisition; images without CAMS file are skipped
  (see the {doc}`CAMS tutorial <../auxdata>` to download it);
- the **DEM** is the GeoTIFF of the tile, if it exists;
- the **L2A product** is named as with the `grs` command line: level name of the image replaced by
  `levname`, extension removed, plus `suffix`, e.g.
  `S2A_MSIL2AGRS_20250801T104041_N0511_R008_T31TEJ_20250801T131318_V2.2.0`. It is a folder of that name
  under `<odir>/<tile>/YYYY/MM/DD/`, with the NetCDF product `<name>.nc`, the ancillary data (CAMS)
  `<name>_anc.nc` and the per-image log file `log_file.log`.

In [ ]:
def output_name(name):
    """Name of the L2A product, as with the grs command line."""
    for level in ('L1C', 'L1GT', 'L1TP'):
        name = name.replace(level, levname)
    return Path(name).stem + (suffix if suffix is not None else f'_V{grs.__version__}')


def dem_file(tile):
    if dem_file_template is None:
        return None
    file = dem_file_template.format(tile=tile)
    return file if os.path.exists(file) else None


def l2a_path(row, date):
    """Output product: <odir>/<tile>/YYYY/MM/DD/<name>/<name>.nc, as with the command line."""
    name = output_name(row['name'])
    return opj(odir, row['tile'], date.strftime('%Y/%m/%d'), name, name + '.nc')


selection = selection.assign(
    cams_file=[cams_file_template.format(date=date) for date in selection.index],
    dem_file=selection['tile'].map(dem_file),
    l2a=[l2a_path(row, date) for date, row in selection.iterrows()])
selection['cams_ok'] = selection['cams_file'].map(os.path.exists)
selection['cloudy'] = selection['cloud_cover'] >= max_cloud_cover
selection['done'] = selection['l2a'].map(os.path.exists)

todo = ~selection['done'] & selection['cams_ok'] & ~selection['cloudy']
print(f"{selection['done'].sum()} already processed, {selection['cloudy'].sum()} too cloudy, "
      f"{(~selection['cams_ok']).sum()} without CAMS file, {todo.sum()} to process")
selection[['name', 'cloud_cover', 'cams_ok', 'dem_file', 'done']]

## 4. Run the batch

For each image:

- it is **skipped** if its L2A output already exists (unless `overwrite=True`), so the batch can be
  interrupted and restarted at any time; it is also skipped if it is too cloudy or has no CAMS file;
- {py:meth}`~grs.grs_process.Process.execute` computes the L2A product and
  {py:meth}`~grs.grs_process.Process.write_output` writes it to the product folder given as `odir`;
- the processing messages are also written to `log_file.log` in the product folder;
- if the processing **fails**, the error is logged, the partially written output (if any) is removed
  so that the image is retried at the next run, and the batch goes on.

In [ ]:
def remove_output(path):
    """Remove an L2A product: NetCDF file and its ancillary file."""
    for p in (path, path.replace('.nc', '_anc.nc')):
        if os.path.isfile(p):
            os.remove(p)


def process_image(row):
    """Process one image; return (status, duration in s, error message)."""
    if not row['cams_ok']:
        return 'no_cams', 0., f"missing {row['cams_file']}"
    if row['cloudy']:
        return 'cloudy', 0., ''
    l2a = row['l2a']
    if os.path.exists(l2a):
        if not overwrite:
            return 'skipped', 0., ''
        remove_output(l2a)
    outdir = os.path.dirname(l2a)
    os.makedirs(outdir, exist_ok=True)

    # copy of the processing messages in the product folder
    handler = logging.FileHandler(opj(outdir, 'log_file.log'))
    handler.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(message)s'))
    logging.getLogger().addHandler(handler)
    t0 = time.perf_counter()
    process = None
    try:
        process = grs.Process()
        process.execute(row['path'], odir=outdir,
                        cams_file=row['cams_file'], dem_file=row['dem_file'],
                        resolution=resolution, opac_model=opac_model, scale_aot=scale_aot)
        process.write_output()
        return 'done', time.perf_counter() - t0, ''
    except Exception as e:
        logging.exception(f"processing failed: {row['path']}")
        remove_output(l2a)
        return 'failed', time.perf_counter() - t0, f'{type(e).__name__}: {e}'
    finally:
        logging.getLogger().removeHandler(handler)
        handler.close()
        # release the memory of the image before the next one
        process = None
        gc.collect()


records = []
for i, (date, row) in enumerate(selection.iterrows(), start=1):
    status, duration, error = process_image(row)
    records.append(dict(date=date, tile=row['tile'], status=status, duration_s=round(duration, 1),
                        error=error, l1=row['path'], l2a=row['l2a']))
    print(f"[{i}/{len(selection)}] {status:7s} {row['name']} ({duration:.0f} s)")

summary = pd.DataFrame(records, columns=['date', 'tile', 'status', 'duration_s', 'error', 'l1', 'l2a'])
summary['status'].value_counts()

The summary is saved as a CSV file in `odir`. Failed images can be inspected below (see also their
`log_file.log`) and reprocessed by running the batch again.

In [ ]:
if len(summary):
    log_file = opj(odir, f'grs_batch_{dt.datetime.now():%Y%m%dT%H%M%S}.csv')
    os.makedirs(odir, exist_ok=True)
    summary.to_csv(log_file, index=False)
    print(f'summary saved to {log_file}')
    processed = summary[summary.status == 'done']
    if len(processed):
        print(f'mean processing time: {processed.duration_s.mean():.0f} s per image')

summary[summary.status.isin(['failed', 'no_cams'])][['date', 'tile', 'status', 'error', 'l1']]

## 5. Quick look at one L2A product

The last available product of the selection is displayed: true-colour composite of the remote-sensing
reflectance $R_{rs}$ of the valid water pixels (`mask == 0`) and aerosol optical thickness at 550 nm
retrieved from the image. The ancillary data (CAMS) are in the `_anc.nc` file next to the product; see the
{doc}`visualization tutorials <../visualization>` for more.

In [ ]:
available = selection['l2a'][selection['l2a'].map(os.path.exists)]
if available.empty:
    raise FileNotFoundError('no L2A product available in the selection: check the summary above')
l2a_file = available.iloc[-1]
l2a = xr.open_dataset(l2a_file, decode_coords='all')
l2a

In [ ]:
water = l2a.mask == 0

# maps in the UTM coordinates (m) of the product
fig, axs = plt.subplots(1, 2, figsize=(18, 8))
l2a.Rrs.sel(wl=[665, 560, 490], method='nearest').where(water).plot.imshow(
    rgb='wl', robust=True, ax=axs[0])
axs[0].set_title('Rrs, valid water pixels')
l2a.aot550.plot.imshow(ax=axs[1], cmap='Spectral_r', cbar_kwargs={'shrink': 0.6, 'label': 'AOT(550)'})
axs[1].set_title('AOT(550) retrieved from the image')
for ax in axs:
    ax.set_aspect('equal')
fig.suptitle(os.path.basename(l2a_file))
plt.tight_layout()

## Alternative: batch from the command line

For long batches, run one `grs` process per image (see {doc}`../../usage`): memory is fully released
after each image. The cell below writes one `grs` command per image of the selection (already processed
images are skipped by `--no_clobber`, too cloudy ones by `--max_cloud_cover`) to a text file, to be run
from a shell. `--resolution` must be given: its default value on the command line is 60 m.

In [ ]:
commands = []
for date, row in selection[selection['cams_ok']].iterrows():
    cmd = (f"grs {row['path']} --cams_file {row['cams_file']} "
           f"--odir {opj(odir, row['tile'], date.strftime('%Y/%m/%d'))} "
           f"--resolution {resolution} --max_cloud_cover {max_cloud_cover} --scale_aot {scale_aot} "
           f"--levname {levname} --no_clobber")
    if row['dem_file']:
        cmd += f" --dem_file {row['dem_file']}"
    if opac_model:
        cmd += f' --opac_model {opac_model}'
    if suffix is not None:
        cmd += f' --suffix {suffix}'
    commands.append(cmd)

command_file = opj(odir, 'grs_commands.txt')
os.makedirs(odir, exist_ok=True)
with open(command_file, 'w') as f:
    f.write('\n'.join(commands) + '\n')
print(f'{len(commands)} commands written to {command_file}\n')
print('\n'.join(commands[:3]))

The commands can then be run one after the other, or several at a time if the memory allows it (up to
30 GB per full Sentinel-2 tile at 20 m with GRS 2.2.0):

```bash
# one after the other
bash /data/satellite/Sentinel-2/L2A_nc/grs_commands.txt

# 2 images in parallel
xargs -P 2 -I{} bash -c '{}' < /data/satellite/Sentinel-2/L2A_nc/grs_commands.txt
```

The command line writes the products with the same names and in the same folders as this notebook, and
its own `log_file.log` and `error.log` in each product folder.